# Gaussian Process Regression on a Light Curve

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
np.random.seed(42)

## generate light curve "data"

We generate the truth *from a GP*, so we know the right answer and can check whether the inference recovers it. The kernel is the squared exponential (RBF):

$$k(t, t') = \sigma_f^2 \exp\!\left[-\frac{(t-t')^2}{2\ell^2}\right]$$

with $\ell = 8$ days, $\sigma_f = 0.30$ mag, and per-epoch noise $\sigma_n = 0.05$ mag.

In [ ]:
ELL_T, SF_T, SN_T = 8.0, 0.30, 0.05        # days, mag, mag  -- the truth


def k_rbf(x, xp, ell, sf):
    return sf ** 2 * np.exp(-0.5 * ((x[:, None] - xp[None, :]) / ell) ** 2)


# light curve with irregular cadence
t = np.concatenate([np.sort(np.random.uniform(0, 45, 26)),
                    np.sort(np.random.uniform(75, 120, 18))])
n = len(t)
tgrid = np.linspace(-10, 140, 400)

# draw truth and data jointly from N(0, K) with a cholesky trick
t_all = np.concatenate([t, tgrid])
L_all = np.linalg.cholesky(k_rbf(t_all, t_all, ELL_T, SF_T) + 1e-9 * np.eye(len(t_all)))
f_all = L_all @ np.random.normal(size=len(t_all))

f_at_t, f_true = f_all[:n], f_all[n:]
y = f_at_t + SN_T * np.random.normal(size=n) # add "observed noise"

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(tgrid, f_true, color='0.75', lw=2, label='true f(t)')
ax.errorbar(t, y, yerr=SN_T, fmt='k.', ms=7, lw=1, label='data')
ax.set_xlabel('time (days)')
ax.set_ylabel('magnitude')
ax.legend(fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

## GP regression
With GP we assume that observed and unobserved function values are **jointly Gaussian**. Write the joint for the training values $\mathbf{y}$ and the values
$\mathbf{f}_* = \mathbf{f}(x_*)$ at any set of test times:

$$\begin{pmatrix}\mathbf{y}\\ \mathbf{f}_*\end{pmatrix} \sim \mathcal{N}\left(
\mathbf{0},\;\begin{pmatrix} K + \sigma_n^2 I & K_* \\ K_*^{\!\top} & K_{**}\end{pmatrix}\right)$$

and condition using the Gaussian conditioning identity:

$$\boxed{\;\bar{\mathbf{f}}_* = K_*^{\!\top}\big(K + \sigma_n^2 I\big)^{-1}\mathbf{y},
\qquad
\mathrm{Cov}[\mathbf{f}_*] = K_{**} - K_*^{\!\top}\big(K + \sigma_n^2 I\big)^{-1}K_*\;}$$

In [ ]:
def gp_predict(t, y, ts, ell, sf, sn):
    ''' Posterior mean and variance of f at times ts given input observation (t, y). 
    
    Returns (mean, sd).
    '''
    K = k_rbf(t, t, ell, sf) + sn ** 2 * np.eye(len(t)) # K+sn^2I

    # cholesky decomposition to invert K+sn^2I
    L = np.linalg.cholesky(K) 
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, y))     # (K + sn^2 I)^-1 y
    
    Ks = k_rbf(t, ts, ell, sf)
    mean = Ks.T @ alpha
    v = np.linalg.solve(L, Ks)
    var = sf ** 2 - np.sum(v ** 2, axis=0)                  # diag(Kss) = sf^2 for the RBF
    return mean, np.sqrt(np.maximum(var, 0.0))


mu, sd = gp_predict(t, y, tgrid, ELL_T, SF_T, SN_T)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.fill_between(tgrid, mu - sd, mu + sd, color='C0', alpha=0.2)
ax.plot(tgrid, mu, 'C0', lw=1.8)
ax.plot(tgrid, f_true, color='0.75', lw=1.5)
ax.errorbar(t, y, yerr=SN_T, fmt='k.', ms=4, lw=0.8)
ax.set_xlabel('time (days)')
ax.set_ylabel('magnitude')
plt.tight_layout()
plt.show()

## varying the kernel
$\ell$ and $\sigma_f$ determine the kernal. Try varying $\ell$ between 0.5 to 60 days. Compare the GP predictions to the observed and true light curves.

## 4. The marginal likelihood
We can integrate out $f$ and write the marginal likelihood as: 
$$p(\mathbf{y}\mid \boldsymbol{\theta}) = \int p(\mathbf{y}\mid \mathbf{f})\,
p(\mathbf{f}\mid\boldsymbol{\theta})\, d\mathbf{f}
= \mathcal{N}(\mathbf{y};\, \mathbf{0},\, K_y), \qquad K_y = K + \sigma_n^2 I$$

$$\log p(\mathbf{y}\mid\boldsymbol{\theta}) =
\underbrace{-\tfrac{1}{2}\mathbf{y}^{\!\top}K_y^{-1}\mathbf{y}}_{\text{data fit}}
\;\underbrace{-\;\tfrac{1}{2}\log|K_y|}_{\text{complexity}}
\;-\;\tfrac{n}{2}\log 2\pi$$

where $\theta = (\ell, \sigma_f)$

In [ ]:
def log_marg_likelihood(t, y, ell, sf):
    ''' log marginal likelihood
    
    Returns (full log marginal likelihood, data-fit term, complexity term).
    ''' 
    K = k_rbf(t, t, ell, sf) + SN_T ** 2 * np.eye(len(t))
    L = np.linalg.cholesky(K)
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, y))
    
    fit = -0.5 * y @ alpha
    complexity = -np.sum(np.log(np.diag(L)))       # = -0.5 log|K_y|
    
    return fit + complexity - 0.5 * len(t) * np.log(2 * np.pi), fit, complexity

explore the log marginal likelihood for a range of $\ell$ and $\sigma_f$

emcee HAMMER it! Implement a reasonable prior and sample the posterior on $\ell, \sigma_f$. Plot the posterior and compare it to the truth

Plot the magnitude-time predictions from the GP using the MAP of your posterior alongside observed data.